# Comparación de iteraciones

Curvas de entrenamiento, tabla comparativa y puntajes de evaluación de todos los runs presentes en `modelos/`. Se regenera después de cada cosecha: cada nuevo run con TensorBoard y `*_evaluacion.json` aparece automáticamente aquí.

In [ ]:
import sys, os
sys.path.append(os.path.abspath("../src"))

import json
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from reportes import extraer_escalares

carpeta_modelos = "../modelos"
runs = []
for ruta_config in sorted(glob.glob(os.path.join(carpeta_modelos, "*", "config.json"))):
    with open(ruta_config, encoding="utf-8") as archivo:
        config = json.load(archivo)
    runs.append(config)

df_configs = pd.DataFrame(runs)
df_configs[["run_id", "algoritmo", "backbone", "dueling", "escala_grises", "pasos", "semilla"]]

## Curvas de recompensa de entrenamiento

`rollout/ep_rew_mean` es la recompensa con clipping de Atari (escala {-1, 0, 1} por paso); sirve para comparar iteraciones entre sí, no para reportar puntaje del juego.

In [ ]:
curvas = {}
for config in runs:
    run_id = config["run_id"]
    try:
        escalares = extraer_escalares(carpeta_modelos, run_id)
        if "rollout/ep_rew_mean" in escalares:
            curvas[run_id] = escalares["rollout/ep_rew_mean"]
        else:
            print(f"{run_id}: sin escalar rollout/ep_rew_mean")
    except Exception as error:
        print(f"{run_id}: {error}")

fig, eje = plt.subplots(figsize=(12, 5))
for run_id, (pasos, valores) in curvas.items():
    eje.plot(pasos, valores, label=run_id)
eje.set_xlabel("Pasos")
eje.set_ylabel("Recompensa media de entrenamiento (clipped)")
eje.set_title("Curvas de entrenamiento por iteración")
eje.legend()
eje.grid(alpha=0.3)
plt.show()

## Evaluación greedy por iteración

Recompensa real del juego (sin clipping), 5 episodios con política greedy, como en la competencia: se toma el máximo de los 5 y también la media.

In [ ]:
filas = []
for ruta in sorted(glob.glob(os.path.join(carpeta_modelos, "*", "*_evaluacion.json"))):
    with open(ruta, encoding="utf-8") as archivo:
        resultado = json.load(archivo)
    filas.append({
        "run_id": os.path.basename(os.path.dirname(ruta)),
        "modelo": os.path.basename(resultado["modelo"]),
        "max": resultado["max"],
        "media": resultado["media"],
        "episodios": resultado["puntajes"],
    })

df_eval = pd.DataFrame(filas)
if df_eval.empty:
    print("Sin evaluaciones registradas aún")
else:
    df_eval.sort_values("max", ascending=False, ignore_index=True)

## Problemas observados por run

Resumen de las entradas de bitácora: inestabilidad, divergencia de Q o estancamiento. Se llena a mano en `bitacora/iteraciones.md`; aquí solo se listan los runs disponibles.

In [ ]:
for config in runs:
    print(config["run_id"], "->", "bitacora/runs/" + config["run_id"] + "/entrada.md")